One sophisticated use of JSON mode is in breaking ambitious tasks into subtasks with JSON, then executing specific workflows for each subtask before fusing them together into a final result.

In this notebook, we'll explore how to use AI task orchestration to plan a 4-day trip to New York City. Breaking down complex tasks into smaller, manageable subtasks is a powerful technique that can help produce more detailed and coherent results.

#### Setup

First, let's import the necessary libraries and set up our connection to the OpenAI API:

<details><summary style="display:list-item; font-size:16px; color:blue;">Jupyter Help</summary>
    
Having trouble testing your work? Double-check that you have followed the steps below to write, run, save, and test your code!
    
[Click here for a walkthrough GIF of the steps below](https://static-assets.codecademy.com/Courses/ds-python/jupyter-help.gif)

Run all initial cells to import libraries and datasets. Then follow these steps for each question:
    
1. Add your solution to the cell with `## YOUR SOLUTION HERE ## `.
2. Run the cell by selecting the `Run` button or the `Shift`+`Enter` keys.
3. Save your work by selecting the `Save` button, the `command`+`s` keys (Mac), or `control`+`s` keys (Windows).
4. Select the `Test Work` button at the bottom left to test your work.

![Screenshot of the buttons at the top of a Jupyter Notebook. The Run and Save buttons are highlighted](https://static-assets.codecademy.com/Paths/ds-python/jupyter-buttons.png)

In [1]:
from openai import OpenAI
import json

client = OpenAI()

def get_completion(prompt, system_prompt="You are a helpful assistant.", json_mode=False):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": prompt}],
        response_format={"type": "json_object"} if json_mode else None
    )
    return response.choices[0].message.content

#### Checkpoint 1/3: Task Planning

To illustrate this point, let's create an agentic workflow for travel planning. Relying on the model's innate knowledge, we'll have it plan out a trip to NYC for a family of four on a budget.

First, we need to break down our complex task into smaller, manageable subtasks. We'll create a function that uses an LLM to analyze the task and return a JSON list of subtasks. To make things more interesting, we'll specify that each list item should be a dict with `text` and `budget` fields.

That way, the initial task will parcel out our total budget to subtasks, giving the subtask prompt useful context.

```python
initial_output = [{
    "text": "Text describing the subtask",
    "budget": "amount of money out of the total travel budget to spend on this task",
}
# and more...
]
```

Let's implement a simple task planning function. In the `<specified_format>` section, add an example of the JSON we should see returned.
In the `<budget>` section of the prompt, specify a family budget of $5,000.
Then, obtain the model output and store it in `json_string`. Load the JSON into Python in the following line, and then assign to `subtasks` the `"data"` field of the result.

In [2]:
def create_subtask_list(prompt):
    return get_completion(prompt, json_mode=True)

prompt = """<task>Your task is to help plan a four day vacation to New York City for a family of four (we live in California.)
You'll do so by providing a list of subtasks involved in the trip.
Try to cover everything we'll need to have great family memories.
Output your result as a JSON object with the key "data" containing a list of subtasks in the format specified below.
Each "content" field should be a description of everything we need to figure out to accomplish this subtask on our trip.
Make the content field about one paragraph long.
</task>

<specified_format>
{{"data": [
{"content": "Subtask description and instructions here",
"budget": "Portion of the budget that can be allocated to this subtask"
},
# and so on
]}}
</specified_format>

<budget>$5,000</budget>

Output ONLY valid JSON below. No markdown backticks.
"""

json_string = create_subtask_list(prompt)
result = json.loads(json_string)
subtasks = result["data"]
print(subtasks)

[{'content': "Research and book roundtrip flights from a major airport in California to New York City for a family of four. Ensure the flight times align with your family's schedule, considering daytime arrivals to maximize your experience. Look for airlines with family-friendly services and compare prices for the best deals. Consider including travel insurance to protect your purchase.", 'budget': '$1,200'}, {'content': 'Book a family-friendly hotel or accommodations in a central location, such as Manhattan. Consider proximity to major attractions, public transportation, and family-oriented amenities like free breakfast, Wi-Fi, and potentially a suite or connected rooms for comfort.', 'budget': '$1,400'}, {'content': 'Plan a daily itinerary that includes iconic NYC attractions like Times Square, Central Park, the Statue of Liberty, the Empire State Building, and the American Museum of Natural History. Make a list of family-friendly attractions and allocate adequate time for exploratio

#### Checkpoint 2/3: Subtask Execution

Now that we have our subtasks, we need to execute each one. We'll create a function that takes the subtasks from Checkpoint 1 and generates a response for each subtask using tailored prompts.

While we're deliberately keeping this very simple, imagine each of these subtasks involving their own sophisticated workflows—by breaking down our more complex task into these sub-units, we can accomplish a great deal.

Initialize an empty list and assign it to `subtask_solutions`. Pass the content and budget of the subtask to the designated fields in the prompt, and then add the model's result to the `subtask_solutions` list.

Then, in the `for` loop, call this function on each item in the subtask list.

In [3]:
subtask_solutions = []
def execute_subtask(subtask):
    prompt = f"""<task>Your task is to help plan the specified aspect of a family of four's four day trip to New York City.
    We're coming from California.
    The task below is one aspect of our overall trip that we want you to plan out for us.
    Output a 2-3 short sentences using your innate knowledge to offer your final recommendation on what to do and purchase.
    We've attached a budget for this domain of our trip, out of a total budget of $5,000.
    Itemize what you'd spend this subsection of the budget on to the best of your knowledge.
    Being your output with a 1-3 word description of your domain, then the 2-3 short sentences and the itemized spending plan (if needed).
    </task>

    <subtask_content>
    {subtask['content']}
    </substask_content>

    <subtask_budget>
    {subtask['budget']}
    </subtask_budget>"""
    result = get_completion(prompt)
    subtask_solutions.append(result)

for subtask in subtasks:
    execute_subtask(subtask)

print(subtask_solutions)

['**Flights**  \nI recommend booking roundtrip flights with a family-friendly airline like Delta, JetBlue, or Southwest for comfort and reliable service. Look for flights departing in the morning from California (e.g., Los Angeles or San Francisco), arriving in New York by mid-afternoon to make the most of your first day. Purchase travel insurance for added peace of mind.\n\n**Itemized Spending Plan:**  \n- Roundtrip economy flights for a family of four: $1,000 (approx. $250 per ticket)  \n- Travel insurance for four: $120  \n- Total: $1,120  \n\nThis leaves a $80 cushion for any booking fees or minor adjustments.', '**Accommodations**  \nFor a comfortable stay in a prime location, book a family-friendly hotel in Midtown Manhattan, close to key attractions and public transportation. Consider options like the Hilton Garden Inn or the Residence Inn by Marriott, which often include free Wi-Fi, breakfast, and family suites. These will provide convenience and amenities within budget.  \n\n*

#### Checkpoint 3/3: Result Synthesis

Now we'll fuse all this verbose output into a single summary with only the most useful information.

We'll begin by concatenating the results of the subtask solutions into a single string.

Then specify the kind of report you'd like to get back from the fusion prompt. Put the details of what you want in your report under the "Output the following" line in the prompt. Some items to consider:

- Offer a detailed, time-based itinerary
- List out all purchases
- Specify locations, times, and contingency plans
- Describe your desired formatting (bullet points, numbered list, markdown headings?)

Finally, call the `fuse_solutions` function and assign it to `final_plan`.

In [4]:
def fuse_solutions(subtask_solutions):
    # Concatenate all subtask solutions with carriage returns between them
    combined_solutions = "\n\n".join(subtask_solutions)
    prompt = f"""<task>Your task is to condense a detailed list of plans for our upcoming trip to NYC into a single, short summary.
    Include only actionable information we can directly use to carry out our vacation.
    Output the following:
    - Offer a detailed, time-based itinerary
    - List out all purchases
    - Specify locations, times, and contingency plans
    - Use bullet point lists
    </task>

    <detailed_plans>
    {combined_solutions}
    </detailed_plans>"""
    return get_completion(prompt)

final_plan = fuse_solutions(subtask_solutions)
print(final_plan)

### Condensed NYC Trip Summary

**Detailed Itinerary:**

- **Day 1: Arrival & Landmarks**  
  - **Morning:** Arrive in NYC by mid-afternoon.  
  - **Afternoon:** Explore Times Square (free), then walk around Central Park (free). Enjoy a quick picnic/snack ($20).  

- **Day 2: Waterfront Views**  
  - **Morning:** Take the Staten Island Ferry (free) to see the Statue of Liberty and skyline views.  
  - **Afternoon:** Relax in Battery Park and grab snacks ($10).  

- **Day 3: Iconic Skyscrapers**  
  - **Morning:** Visit the Empire State Building ($84 for tickets).  
  - **Afternoon:** Walk around Bryant Park; grab coffee/snacks ($10).  

- **Day 4: Museum & Leisure**  
  - **Morning:** Visit the American Museum of Natural History ($40 family tickets).  
  - **Afternoon:** Explore nearby shops/parks, and use downtime for rest.  

**Purchases Summary:**

- **Flights:** Roundtrip for 4, ~$1,000; travel insurance $120.  
- **Accommodations:** 4 nights in Midtown (~$1,400 including taxes/fee